# Description

In this notebook, we benchmark PySR algorithm on the set of previously generated SR benchmarks.

In [1]:
from __future__ import annotations

import csv
import time
from pathlib import Path

import h5py
import numpy as np
import sympy as sp

from pysr import PySRRegressor

from config.benchmark_config import DataCFG, PYSR


def _load_one_group(f: h5py.File, gname: str):
    g = f[gname]
    raw = g["sympy_str"][()]
    true_expr_str = raw.decode("utf-8") if isinstance(raw, (bytes, bytearray)) else str(raw)

    Xtr = g["train"]["X"][...].astype(np.float64, copy=False)
    ytr = g["train"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    Xti = g["test_interp"]["X"][...].astype(np.float64, copy=False)
    yti = g["test_interp"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    Xte = g["test_extrap"]["X"][...].astype(np.float64, copy=False)
    yte = g["test_extrap"]["y"][...].astype(np.float64, copy=False).reshape(-1)

    return true_expr_str, Xtr, ytr, Xti, yti, Xte, yte


def _mse(yhat: np.ndarray, y: np.ndarray) -> float:
    yhat = np.asarray(yhat, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    return float(np.mean((yhat - y) ** 2))


def _feature_names(n_features: int) -> list[str]:
    # Match your SymPy convention: x1, x2, ...
    return [f"x{i+1}" for i in range(n_features)]


def main():
    cfg = DataCFG()

    out_csv = Path(PYSR.results_path)
    out_csv.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(cfg.h5_path, "r") as f, out_csv.open("w", newline="") as out:
        w = csv.writer(out)
        w.writerow(
            [
                "group",
                "run",
                "seed",
                "n_train",
                "n_features",
                "train_mse",
                "test_interp_mse",
                "test_extrap_mse",
                "duration_s",
                "true_expr",
                "found_expr",
            ]
        )

        groups = sorted(f.keys())
        print(groups)

        for gname in groups:
            true_expr_str, Xtr, ytr, Xti, yti, Xte, yte = _load_one_group(f, gname)

            n_features = Xtr.shape[1]
            feature_names = _feature_names(n_features)

            successful_runs = 0
            attempt = 0

            while successful_runs < PYSR.n_runs:
                seed = PYSR.base_seed + attempt
                attempt += 1

                model = PySRRegressor(
                    niterations=PYSR.niterations,
                    populations=PYSR.populations,
                    maxsize=PYSR.maxsize,
                    timeout_in_seconds=PYSR.timeout_in_seconds,
                    unary_operators=list(PYSR.unary_ops),
                    binary_operators=list(PYSR.binary_ops),
                    elementwise_loss=PYSR.elementwise_loss,
                    model_selection=PYSR.model_selection,
                    verbosity=PYSR.verbosity,
                    progress=PYSR.progress,
                    temp_equation_file=PYSR.temp_equation_file,
                    delete_tempfiles=PYSR.delete_tempfiles,
                    random_state=seed,
                    deterministic=True,
                    parallelism="serial",
                )

                t0 = time.perf_counter()
                model.fit(Xtr, ytr, variable_names=feature_names)
                dur = time.perf_counter() - t0

                yhat_tr = model.predict(Xtr)
                yhat_ti = model.predict(Xti)
                yhat_te = model.predict(Xte)

                train_mse = _mse(yhat_tr, ytr)
                test_interp_mse = _mse(yhat_ti, yti)
                test_extrap_mse = _mse(yhat_te, yte)

                # 🔒 HARD FILTER: reject non-finite runs
                if not (
                    np.isfinite(train_mse)
                    and np.isfinite(test_interp_mse)
                    and np.isfinite(test_extrap_mse)
                ):
                    print(
                        f"[{gname}] seed={seed} FAILED "
                        f"(non-finite MSEs) → retry"
                    )
                    continue

                found_expr_str = ""
                try:
                    expr = model.sympy()
                    found_expr_str = "" if expr is None else str(expr)
                except Exception:
                    pass

                w.writerow(
                    [
                        gname,
                        successful_runs,
                        seed,
                        int(Xtr.shape[0]),
                        n_features,
                        train_mse,
                        test_interp_mse,
                        test_extrap_mse,
                        dur,
                        true_expr_str,
                        found_expr_str,
                    ]
                )
                out.flush()

                print(
                    f"[{gname}] run={successful_runs} seed={seed} "
                    f"train={train_mse:.3e} interp={test_interp_mse:.3e} "
                    f"extrap={test_extrap_mse:.3e} dur={dur:.2f}s"
                )
                if found_expr_str:
                    print("Found:", found_expr_str)

                successful_runs += 1

    print(f"\nSaved: {out_csv}")


if __name__ == "__main__":
    main()

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
['expr_000_lin_uni', 'expr_001_lin_bi', 'expr_002_poly2_uni', 'expr_003_poly2_bi', 'expr_004_log_poly2_sign2', 'expr_005_sqrt_poly2_sign2', 'expr_006_rat_linlin_uni_pole_train', 'expr_007_rat_linlin_bi_pole_train', 'expr_008_rat_poly2poly2_uni_one_pole_train', 'expr_009_rat_poly2poly2_uni_two_poles_train']
[expr_000_lin_uni] run=0 seed=0 train=4.699e-32 interp=3.939e-32 extrap=2.274e-31 dur=358.76s
Found: x1*0.8340862 + x1*1.0359138 + 2.01
[expr_000_lin_uni] run=1 seed=1 train=3.763e-15 interp=6.391e-15 extrap=3.852e-18 dur=358.12s
Found: -(-0.9241726)*x1 + (x1 + x1)*0.4729137 + 2.01 - 4.0603934e-9/(1.1479187 - x1)
[expr_000_lin_uni] run=2 seed=2 train=5.533e-16 interp=5.169e-16 extrap=3.612e-15 dur=336.13s
Found: x1*0.0042837197 + x1*0.8657163 + x1 + 2.01
[expr_000_lin_uni] run=3 seed=3 train=1.283e-15 interp=1.199e-15 extrap=8.377e-15 dur=343.29s
Found: -0.07338223*x1 + x

In [2]:
from __future__ import annotations

import math
from pathlib import Path

import pandas as pd
import sympy as sp


def _safe_latex(expr_str: str) -> str | None:
    if not isinstance(expr_str, str) or not expr_str.strip():
        return None
    try:
        expr = sp.sympify(expr_str)
        return sp.latex(expr)
    except Exception:
        return None


def _format_pm(value: float, std: float, sig: int = 1) -> str:
    if value == 0.0:
        return r"(0\pm0)\times 10^{0}"

    exp = int(math.floor(math.log10(abs(value))))
    scale = 10 ** exp

    v = round(value / scale, sig)
    s = round(std / scale, sig)

    return rf"({v}\pm{s})\times 10^{{{exp}}}"


def _count_nodes(expr: sp.Expr) -> int:
    return sum(1 for _ in sp.preorder_traversal(expr))


def summarize(csv_path: str | Path, k: int = 5) -> None:
    df = pd.read_csv(csv_path)

    metrics = [
        "train_mse",
        "test_interp_mse",
        "test_extrap_mse",
    ]

    for gname, gdf in df.groupby("group"):
        print(f"\n{gname}")

        # take k best by extrapolation error (same criterion as CEQL)
        gdf = gdf.sort_values("test_extrap_mse").iloc[:k]

        # ---- errors ----
        for m in metrics:
            vals = gdf[m].astype(float).to_numpy()
            mean = float(vals.mean())
            std = float(vals.std(ddof=0))
            print(f"  {m}: {_format_pm(mean, std)}")

        # ---- symbolic complexity ----
        node_counts = []
        for s in gdf["found_expr"]:
            if not isinstance(s, str) or not s.strip():
                continue
            try:
                expr = sp.sympify(s)
                node_counts.append(_count_nodes(expr))
            except Exception:
                pass

        if node_counts:
            nc = pd.Series(node_counts, dtype=float)
            print(
                f"  expr_nodes: "
                f"{nc.mean():.1f} ± {nc.std(ddof=0):.1f}"
            )
        else:
            print("  expr_nodes: N/A")

        # ---- best train-fit expression (same as CEQL) ----
        best_row = gdf.sort_values("train_mse").iloc[0]
        latex_expr = _safe_latex(best_row["found_expr"])

        if latex_expr is not None:
            print("  best_train_expr_latex:")
            print(f"    ${latex_expr}$")
        else:
            print("  best_train_expr_latex: N/A")


if __name__ == "__main__":
    summarize("reports/sr_benchmark_pysr.csv", k=5)


expr_000_lin_uni
  train_mse: (1.1\pm1.4)\times 10^{-15}
  test_interp_mse: (1.6\pm2.4)\times 10^{-15}
  test_extrap_mse: (2.4\pm3.3)\times 10^{-15}
  expr_nodes: 6.8 ± 3.6
  best_train_expr_latex:
    $1.87 x_{1} + 2.01$

expr_001_lin_bi
  train_mse: (6.4\pm7.9)\times 10^{-15}
  test_interp_mse: (6.0\pm7.3)\times 10^{-15}
  test_extrap_mse: (1.7\pm2.1)\times 10^{-14}
  expr_nodes: 8.0 ± 0.0
  best_train_expr_latex:
    $1.56 x_{1} + 1.59 x_{2} - 2.91$

expr_002_poly2_uni
  train_mse: (4.3\pm4.4)\times 10^{-15}
  test_interp_mse: (4.6\pm4.7)\times 10^{-15}
  test_extrap_mse: (2.7\pm3.2)\times 10^{-14}
  expr_nodes: 13.0 ± 1.1
  best_train_expr_latex:
    $1.6543639 x_{1} + 2.48 \left(x_{1} - 0.00018166579\right) \left(x_{1} + 0.107292995\right) - 0.67995167$

expr_003_poly2_bi
  train_mse: (5.1\pm4.5)\times 10^{-9}
  test_interp_mse: (5.8\pm5.5)\times 10^{-9}
  test_extrap_mse: (2.4\pm2.7)\times 10^{-7}
  expr_nodes: 19.6 ± 0.5
  best_train_expr_latex:
    $- x_{1} \left(- 0.041307323